# EPIC Clarity Measurement Hydration

This notebook hydrates the OMOP MEASUREMENT table from EPIC Clarity lab results and vital signs.

## Source Tables
- `_exponent._bronze_epic_clarity.order_results` - Lab/result findings
- `_exponent._bronze_epic_clarity.clarity_component` - Component master with LOINC codes

## Concept Mapping Strategy
- Join `order_results.COMPONENT_ID` → `clarity_component.COMPONENT_ID` to get LOINC codes
- Map LOINC → standard Measurement concept via `concept_relationship`
- Map `REFERENCE_UNIT` → UCUM unit concepts via `concept` table

## OMOP Fields Populated
- measurement_id (surrogate key)
- measurement_source_value (COMPONENT_ID or LOINC code)
- measurement_concept_id (mapped from LOINC codes)
- measurement_source_concept_id (source LOINC concept)
- measurement_date
- value_source_value
- unit_source_value
- unit_concept_id (mapped from REFERENCE_UNIT)
- visit_occurrence_id (if available)

In [ ]:
%sql
-- TRUNCATE Gold table for Epic (run this to clear stale data before reload)
TRUNCATE TABLE _exponent.omop_epic.measurement;

In [ ]:
%sql
-- Delete Epic records from Silver and Mapping tables (for full refresh)
DELETE FROM _exponent.omop_silver.measurement WHERE source_system = 'epic_clarity';

DELETE FROM _exponent.omop_mapping.source_to_measurement WHERE source_system = 'epic_clarity';

In [ ]:
source = 'epic_clarity'

In [ ]:
%sql
-- Create standard concept mapping view for LOINC → Measurement concepts
-- Maps LOINC codes to standard Measurement domain concepts
CREATE OR REPLACE TEMPORARY VIEW standard_concept_mapping AS
WITH ranked AS (
  SELECT
    concept.vocabulary_id         AS source_vocabulary_id,
    concept.concept_code          AS source_concept_code,
    concept.concept_id            AS source_concept_id,
    standard_concept.concept_id   AS standard_concept_id,
    standard_concept.concept_name AS standard_concept_name,
    concept_relationship.valid_start_date AS rel_valid_start_date,
    ROW_NUMBER() OVER (
      PARTITION BY concept.vocabulary_id, concept.concept_code
      ORDER BY concept_relationship.valid_start_date DESC, standard_concept.concept_id ASC
    ) AS rn
  FROM _exponent.omop.concept
  JOIN _exponent.omop.concept_relationship
    ON concept_relationship.concept_id_1 = concept.concept_id
   AND concept_relationship.relationship_id = 'Maps to'
   AND concept_relationship.invalid_reason IS NULL
  JOIN _exponent.omop.concept AS standard_concept
    ON standard_concept.concept_id = concept_relationship.concept_id_2
   AND standard_concept.standard_concept = 'S'
   AND standard_concept.domain_id = 'Measurement'
   AND standard_concept.invalid_reason IS NULL
  WHERE concept.vocabulary_id = 'LOINC'
    AND concept.invalid_reason IS NULL
)
SELECT
  source_vocabulary_id,
  source_concept_code,
  source_concept_id,
  standard_concept_id,
  standard_concept_name
FROM ranked
WHERE rn = 1;

In [ ]:
%sql
-- Create unit concept mapping view for unit strings → UCUM concepts
-- Maps common unit strings to OMOP Unit domain concepts
CREATE OR REPLACE TEMPORARY VIEW unit_concept_mapping AS
SELECT
  concept_code AS unit_source_value_cleaned,
  concept_id   AS unit_concept_id,
  concept_name AS unit_concept_name
FROM _exponent.omop.concept
WHERE vocabulary_id = 'UCUM'
  AND standard_concept = 'S'
  AND domain_id = 'Unit'
  AND invalid_reason IS NULL

UNION ALL

-- Add common unit string variations that don't exactly match UCUM
SELECT 'mg/dL' AS unit_source_value_cleaned, 8840 AS unit_concept_id, 'milligram per deciliter' AS unit_concept_name
UNION ALL SELECT 'mg/dl', 8840, 'milligram per deciliter'
UNION ALL SELECT 'MG/DL', 8840, 'milligram per deciliter'
UNION ALL SELECT 'g/dL', 8713, 'gram per deciliter'
UNION ALL SELECT 'g/dl', 8713, 'gram per deciliter'
UNION ALL SELECT 'G/DL', 8713, 'gram per deciliter'
UNION ALL SELECT 'mmol/L', 8753, 'millimole per liter'
UNION ALL SELECT 'mmol/l', 8753, 'millimole per liter'
UNION ALL SELECT 'MMOL/L', 8753, 'millimole per liter'
UNION ALL SELECT 'mEq/L', 9557, 'milliequivalent per liter'
UNION ALL SELECT 'meq/L', 9557, 'milliequivalent per liter'
UNION ALL SELECT 'MEQ/L', 9557, 'milliequivalent per liter'
UNION ALL SELECT 'U/L', 8645, 'unit per liter'
UNION ALL SELECT 'u/L', 8645, 'unit per liter'
UNION ALL SELECT 'IU/L', 8923, 'international unit per liter'
UNION ALL SELECT 'IU/l', 8923, 'international unit per liter'
UNION ALL SELECT 'ng/mL', 8842, 'nanogram per milliliter'
UNION ALL SELECT 'ng/ml', 8842, 'nanogram per milliliter'
UNION ALL SELECT 'NG/ML', 8842, 'nanogram per milliliter'
UNION ALL SELECT 'pg/mL', 8845, 'picogram per milliliter'
UNION ALL SELECT 'pg/ml', 8845, 'picogram per milliliter'
UNION ALL SELECT 'cells/uL', 8647, 'cells per microliter'
UNION ALL SELECT 'cells/mcL', 8647, 'cells per microliter'
UNION ALL SELECT '/uL', 8647, 'per microliter'
UNION ALL SELECT '/mcL', 8647, 'per microliter'
UNION ALL SELECT '%', 8554, 'percent'
UNION ALL SELECT 'percent', 8554, 'percent'
UNION ALL SELECT 'PERCENT', 8554, 'percent'
UNION ALL SELECT 'kg', 9529, 'kilogram'
UNION ALL SELECT 'KG', 9529, 'kilogram'
UNION ALL SELECT 'lbs', 8739, 'pound'
UNION ALL SELECT 'LBS', 8739, 'pound'
UNION ALL SELECT 'lb', 8739, 'pound'
UNION ALL SELECT 'cm', 8582, 'centimeter'
UNION ALL SELECT 'CM', 8582, 'centimeter'
UNION ALL SELECT 'in', 9330, 'inch'
UNION ALL SELECT 'IN', 9330, 'inch'
UNION ALL SELECT 'inches', 9330, 'inch'
UNION ALL SELECT 'mmHg', 8876, 'millimeter mercury column'
UNION ALL SELECT 'mm Hg', 8876, 'millimeter mercury column'
UNION ALL SELECT 'MMHG', 8876, 'millimeter mercury column'
UNION ALL SELECT 'bpm', 8541, 'per minute'
UNION ALL SELECT 'BPM', 8541, 'per minute'
UNION ALL SELECT '/min', 8541, 'per minute'
UNION ALL SELECT 'beats/min', 8541, 'per minute'
UNION ALL SELECT 'breaths/min', 8541, 'per minute';

In [ ]:
%sql
-- Create silver_measurement temp view for Epic Clarity
-- Now joins to clarity_component to get LOINC codes for concept mapping
CREATE OR REPLACE TEMPORARY VIEW silver_measurement AS

SELECT
    stp.person_id,
    -- Map LOINC code to standard measurement concept
    COALESCE(scm.standard_concept_id, 0) AS measurement_concept_id,
    DATE(ore.RESULT_DATE) AS measurement_date,
    ore.RESULT_DATE AS measurement_datetime,
    NULL AS measurement_time,
    32817 AS measurement_type_concept_id,  -- EHR
    0 AS operator_concept_id,
    TRY_CAST(ore.ORD_VALUE AS DOUBLE) AS value_as_number,
    0 AS value_as_concept_id,
    -- Map unit string to UCUM concept
    COALESCE(ucm.unit_concept_id, 0) AS unit_concept_id,
    TRY_CAST(ore.REFERENCE_LOW AS DOUBLE) AS range_low,
    TRY_CAST(ore.REFERENCE_HIGH AS DOUBLE) AS range_high,
    NULL AS provider_id,
    NULL AS visit_occurrence_id,
    NULL AS visit_detail_id,
    -- Store LOINC code as measurement_source_value (more useful than COMPONENT_ID)
    COALESCE(cc.LOINC_CODE, CAST(ore.COMPONENT_ID AS STRING)) AS measurement_source_value,
    -- Source concept is the LOINC concept itself (not mapped)
    COALESCE(scm.source_concept_id, 0) AS measurement_source_concept_id,
    ore.REFERENCE_UNIT AS unit_source_value,
    0 AS unit_source_concept_id,
    ore.ORD_VALUE AS value_source_value,
    NULL AS measurement_event_id,
    NULL AS meas_event_field_concept_id,
    CONCAT_WS(CHR(31), 'epic_clarity', 'ORDER_RESULTS', 'ORDER_PROC_ID', CAST(ore.ORDER_PROC_ID AS STRING), 'LINE', CAST(ore.LINE AS STRING)) AS measurement_unique_key,
    'epic_clarity' AS source_system
FROM _exponent._bronze_epic_clarity.order_results ore

-- Join to clarity_component to get LOINC codes
LEFT JOIN _exponent._bronze_epic_clarity.clarity_component cc
    ON CAST(ore.COMPONENT_ID AS DECIMAL) = cc.COMPONENT_ID

INNER JOIN _exponent._bronze_epic_clarity.pat_enc pe
    ON ore.PAT_ENC_CSN_ID = pe.PAT_ENC_CSN_ID

INNER JOIN _exponent.omop_mapping.source_to_person stp
    ON stp.person_source_value = CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', pe.PAT_ID)
    AND stp.active_flag = TRUE

-- Map LOINC code to standard measurement concept
LEFT JOIN standard_concept_mapping scm
    ON scm.source_concept_code = cc.LOINC_CODE
    AND scm.source_vocabulary_id = 'LOINC'

-- Map unit string to UCUM concept
LEFT JOIN unit_concept_mapping ucm
    ON TRIM(ore.REFERENCE_UNIT) = ucm.unit_source_value_cleaned

WHERE ore.ORDER_PROC_ID IS NOT NULL
    AND ore.RESULT_DATE IS NOT NULL
    AND pe.PAT_ID IS NOT NULL

In [ ]:
%sql
-- Merge to Silver layer
MERGE INTO _exponent.omop_silver.measurement AS t
USING (
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY measurement_unique_key
                ORDER BY measurement_date DESC
            ) AS rn
        FROM silver_measurement
    ) WHERE rn = 1
) AS s
ON t.measurement_source_value = s.measurement_unique_key

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.measurement_concept_id <=> s.measurement_concept_id)
  OR NOT (t.measurement_date <=> s.measurement_date)
  OR NOT (t.measurement_datetime <=> s.measurement_datetime)
  OR NOT (t.measurement_type_concept_id <=> s.measurement_type_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.value_source_value <=> s.value_source_value)
)
THEN UPDATE SET
    t.person_id = s.person_id,
    t.measurement_concept_id = s.measurement_concept_id,
    t.measurement_date = s.measurement_date,
    t.measurement_datetime = s.measurement_datetime,
    t.measurement_time = s.measurement_time,
    t.measurement_type_concept_id = s.measurement_type_concept_id,
    t.operator_concept_id = s.operator_concept_id,
    t.value_as_number = s.value_as_number,
    t.value_as_concept_id = s.value_as_concept_id,
    t.unit_concept_id = s.unit_concept_id,
    t.range_low = s.range_low,
    t.range_high = s.range_high,
    t.provider_id = s.provider_id,
    t.visit_occurrence_id = s.visit_occurrence_id,
    t.visit_detail_id = s.visit_detail_id,
    t.measurement_source_concept_id = s.measurement_source_concept_id,
    t.unit_source_value = s.unit_source_value,
    t.unit_source_concept_id = s.unit_source_concept_id,
    t.value_source_value = s.value_source_value,
    t.source_system = s.source_system,
    t.last_mod_tsp = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    person_id,
    measurement_concept_id,
    measurement_date,
    measurement_datetime,
    measurement_time,
    measurement_type_concept_id,
    operator_concept_id,
    value_as_number,
    value_as_concept_id,
    unit_concept_id,
    range_low,
    range_high,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    measurement_source_value,
    measurement_source_concept_id,
    unit_source_value,
    unit_source_concept_id,
    value_source_value,
    source_system,
    last_mod_tsp
)
VALUES (
    s.person_id,
    s.measurement_concept_id,
    s.measurement_date,
    s.measurement_datetime,
    s.measurement_time,
    s.measurement_type_concept_id,
    s.operator_concept_id,
    s.value_as_number,
    s.value_as_concept_id,
    s.unit_concept_id,
    s.range_low,
    s.range_high,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.measurement_unique_key,
    s.measurement_source_concept_id,
    s.unit_source_value,
    s.unit_source_concept_id,
    s.value_source_value,
    s.source_system,
    CURRENT_TIMESTAMP()
)

In [ ]:
%sql
-- Insert new mappings to source_to_measurement
INSERT INTO _exponent.omop_mapping.source_to_measurement (
    source_system,
    measurement_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp
)
SELECT
    s.source_system,
    s.measurement_source_value,
    s.person_id,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.last_mod_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp
FROM (
    SELECT DISTINCT source_system, measurement_source_value, person_id, last_mod_tsp
    FROM _exponent.omop_silver.measurement
    WHERE source_system = 'epic_clarity'
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement x
    ON s.measurement_source_value = x.measurement_source_value

In [ ]:
%sql
-- Create gold_measurement temp view
CREATE OR REPLACE TEMPORARY VIEW gold_measurement AS
SELECT
    sm.measurement_id,
    s.person_id,
    s.measurement_concept_id,
    s.measurement_date,
    s.measurement_datetime,
    s.measurement_time,
    s.measurement_type_concept_id,
    s.operator_concept_id,
    s.value_as_number,
    s.value_as_concept_id,
    s.unit_concept_id,
    s.range_low,
    s.range_high,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.measurement_source_value,
    s.measurement_source_concept_id,
    s.unit_source_value,
    s.unit_source_concept_id,
    s.value_source_value
FROM _exponent.omop_silver.measurement s
JOIN _exponent.omop_mapping.source_to_measurement sm
    ON sm.measurement_source_value = s.measurement_source_value
    AND sm.active_flag = TRUE
-- Ensure person_id exists in gold person table (prevents orphan records)
INNER JOIN _exponent.omop_epic.person p
    ON p.person_id = s.person_id
WHERE s.source_system = 'epic_clarity'
    AND s.person_id IS NOT NULL
    -- Exclude measurements before birth (plausibility check)
    AND s.measurement_date >= DATE(p.birth_datetime)

In [ ]:
%sql
-- Merge to Gold layer (omop_epic)
MERGE INTO _exponent.omop_epic.measurement AS gold
USING gold_measurement AS src
ON gold.measurement_id = src.measurement_id

WHEN MATCHED THEN UPDATE SET
    gold.person_id = src.person_id,
    gold.measurement_concept_id = src.measurement_concept_id,
    gold.measurement_date = src.measurement_date,
    gold.measurement_datetime = src.measurement_datetime,
    gold.measurement_time = src.measurement_time,
    gold.measurement_type_concept_id = src.measurement_type_concept_id,
    gold.operator_concept_id = src.operator_concept_id,
    gold.value_as_number = src.value_as_number,
    gold.value_as_concept_id = src.value_as_concept_id,
    gold.unit_concept_id = src.unit_concept_id,
    gold.range_low = src.range_low,
    gold.range_high = src.range_high,
    gold.provider_id = src.provider_id,
    gold.visit_occurrence_id = src.visit_occurrence_id,
    gold.visit_detail_id = src.visit_detail_id,
    gold.measurement_source_value = src.measurement_source_value,
    gold.measurement_source_concept_id = src.measurement_source_concept_id,
    gold.unit_source_value = src.unit_source_value,
    gold.unit_source_concept_id = src.unit_source_concept_id,
    gold.value_source_value = src.value_source_value

WHEN NOT MATCHED THEN INSERT (
    measurement_id,
    person_id,
    measurement_concept_id,
    measurement_date,
    measurement_datetime,
    measurement_time,
    measurement_type_concept_id,
    operator_concept_id,
    value_as_number,
    value_as_concept_id,
    unit_concept_id,
    range_low,
    range_high,
    provider_id,
    visit_occurrence_id,
    visit_detail_id,
    measurement_source_value,
    measurement_source_concept_id,
    unit_source_value,
    unit_source_concept_id,
    value_source_value
)
VALUES (
    src.measurement_id,
    src.person_id,
    src.measurement_concept_id,
    src.measurement_date,
    src.measurement_datetime,
    src.measurement_time,
    src.measurement_type_concept_id,
    src.operator_concept_id,
    src.value_as_number,
    src.value_as_concept_id,
    src.unit_concept_id,
    src.range_low,
    src.range_high,
    src.provider_id,
    src.visit_occurrence_id,
    src.visit_detail_id,
    src.measurement_source_value,
    src.measurement_source_concept_id,
    src.unit_source_value,
    src.unit_source_concept_id,
    src.value_source_value
)

In [ ]:
%sql
-- Validation queries
-- 1. Check person_id FK integrity (should be 0 orphan records)
SELECT 'MEASUREMENT.PERSON_ID FK' as check_field,
       COUNT(*) as orphan_records
FROM _exponent.omop_epic.measurement m
WHERE NOT EXISTS (SELECT 1 FROM _exponent.omop_epic.person p WHERE p.person_id = m.person_id);

-- 2. Total record count
SELECT 'total_records' as check_field, COUNT(*) as cnt FROM _exponent.omop_epic.measurement;

-- 3. Measurement concept mapping rate (should be > 0% now)
SELECT 'measurement_concept_id mapping' as check_field,
       COUNT(*) as total,
       SUM(CASE WHEN measurement_concept_id = 0 THEN 1 ELSE 0 END) as unmapped,
       SUM(CASE WHEN measurement_concept_id > 0 THEN 1 ELSE 0 END) as mapped,
       ROUND(SUM(CASE WHEN measurement_concept_id > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_mapped
FROM _exponent.omop_epic.measurement;

-- 4. Unit concept mapping rate (should be > 0% now)
SELECT 'unit_concept_id mapping' as check_field,
       COUNT(*) as total,
       SUM(CASE WHEN unit_concept_id = 0 THEN 1 ELSE 0 END) as unmapped,
       SUM(CASE WHEN unit_concept_id > 0 THEN 1 ELSE 0 END) as mapped,
       ROUND(SUM(CASE WHEN unit_concept_id > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_mapped
FROM _exponent.omop_epic.measurement;

-- 5. Top 10 unmapped LOINC codes (for future mapping work)
SELECT measurement_source_value as loinc_code, COUNT(*) as cnt
FROM _exponent.omop_epic.measurement
WHERE measurement_concept_id = 0
GROUP BY measurement_source_value
ORDER BY cnt DESC
LIMIT 10;

-- 6. Top 10 unmapped units (for future mapping work)
SELECT unit_source_value, COUNT(*) as cnt
FROM _exponent.omop_epic.measurement
WHERE unit_concept_id = 0 AND unit_source_value IS NOT NULL
GROUP BY unit_source_value
ORDER BY cnt DESC
LIMIT 10;